In [1]:
import torchvision

print(torchvision.__version__)

0.22.0+cu118


In [2]:
import torch
from torch import optim as optim
from torchvision.models.detection import RetinaNet
from torchvision.models.detection.retinanet import RetinaNetHead
from torchvision.models import mobilenet_v3_large
from torchvision.models.detection.backbone_utils import IntermediateLayerGetter, FeaturePyramidNetwork

In [15]:
from torchvision.models.detection import RetinaNet
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.backbone_utils import _validate_trainable_layers
from torchvision.models import mobilenet_v3_large
from torchvision.ops import FeaturePyramidNetwork

# 1️⃣ Create a backbone
mobilenet_backbone = mobilenet_v3_large(weights="IMAGENET1K_V1").features

# Select layers that produce good feature maps for FPN
backbone_out_channels = [40, 112, 160]  # example MobileNetV3 feature map channels
fpn = FeaturePyramidNetwork(in_channels_list=backbone_out_channels, out_channels=256)

class MobileNetBackbone(torch.nn.Module):
    def __init__(self, body, fpn):
        super().__init__()
        self.body = body
        self.fpn = fpn

    def forward(self, x):
        features = []
        for i, block in enumerate(self.body):
            x = block(x)
            # Pick some layers for FPN
            if i in {3, 6, 12}:  # adjust indices
                features.append(x)
        # Create dict for FPN
        features = {str(idx): f for idx, f in enumerate(features)}
        return self.fpn(features)

    @property
    def out_channels(self):
        return 256

backbone = MobileNetBackbone(mobilenet_backbone, fpn)

# 2️⃣ Define anchor generator matching the number of FPN outputs
# If FPN outputs 3 feature maps, you need 3 tuples here
anchor_generator = AnchorGenerator(
    sizes=((32,), (64,), (128,)),       # one tuple per feature map
    aspect_ratios=((0.5, 1.0, 2.0),) * 3
)

# 3️⃣ Create RetinaNet model
model = RetinaNet(backbone, num_classes=2, anchor_generator=anchor_generator)
model.to(device)


RetinaNet(
  (backbone): MobileNetBackbone(
    (body): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          )
        )
      )
      (2): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivati

In [16]:
if torch.cuda.is_available():
    device = torch.device("cuda:0") # Use the first GPU
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Training on CPU.")

Training on GPU: NVIDIA GeForce RTX 2060 SUPER


In [17]:
import torch
from torchvision.datasets import CocoDetection
from torchvision import transforms as T
import os

class CocoDetectionRetinaNet(CocoDetection):
    def __init__(self, root, annFile, transform=None):
        super(CocoDetectionRetinaNet, self).__init__(root, annFile)
        self.transform = transform

    def __getitem__(self, idx):
        img, ann = super().__getitem__(idx)

        boxes = []
        labels = []

        for obj in ann:
            if 'iscrowd' in obj and obj['iscrowd']:
                continue

            bbox = obj['bbox']
            x1 = bbox[0]
            y1 = bbox[1]
            x2 = bbox[0] + bbox[2]
            y2 = bbox[1] + bbox[3]

            boxes.append([x1, y1, x2, y2])
            labels.append(obj['category_id'])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        if self.transform is not None:
            img = self.transform(img)

        return img, target


In [18]:
!pip install pycocotools

In [19]:
from torchvision.datasets import CocoDetection
from torchvision.transforms import ToTensor

train_ds = CocoDetectionRetinaNet(root='cocoshrimpdataset/train',
                         annFile='cocoshrimpdataset/annotations/_annotations_train.coco.json',
                         transform=ToTensor())

test_ds = CocoDetectionRetinaNet(root='cocoshrimpdataset/test',
                        annFile='cocoshrimpdataset/annotations/_annotations_test.coco.json',
                        transform=ToTensor())

val_ds = CocoDetectionRetinaNet(root='cocoshrimpdataset/valid',
                       annFile='cocoshrimpdataset/annotations/_annotations_valid.coco.json',
                       transform=ToTensor())

loading annotations into memory...
Done (t=0.42s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.12s)
creating index...
index created!


In [20]:
from torch.utils.data import DataLoader

train_dl = DataLoader(train_ds,
                      batch_size=8,
                      shuffle=True,
                      collate_fn=lambda batch: tuple(zip(*batch)))

test_dl = DataLoader(test_ds,
                     batch_size=8,
                     shuffle=False,
                     collate_fn=lambda batch: tuple(zip(*batch)))

val_dl = DataLoader(val_ds,
                    batch_size=8,
                    shuffle=False,
                    collate_fn=lambda batch: tuple(zip(*batch)))

In [ ]:
import torch
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ✅ Make sure your model is on the correct device
model.to(device)

# ✅ Define optimizer *after* model is on device
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, targets in tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True):
        # Move images and targets to GPU/CPU
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Skip empty annotations (no boxes)
        if any(t['boxes'].numel() == 0 for t in targets):
            continue

        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backpropagation
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item()

    avg_loss = running_loss / len(train_dl)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f}")

    # ✅ Save model weights safely
    save_path = f"runs/retinanet_mobilenetv3_fpn_epoch{epoch+1}.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Saved weights to {save_path}")


Using device: cuda


Epoch 1/20:   0%|          | 0/518 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 314.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 14.18 GiB is allocated by PyTorch, and 510.24 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)